# NeoPZ H(div): archived comparison results

This notebook inspects 46 recorded comparisons between pyMHM and NeoPZ RT0/P0. The C++ execution driver was `neopz_rt0`, using NeoPZ's EHDivConstant family at order zero. The full trace has two constant segments per macroedge; the restricted trace has one. Both have two fine segments per macroedge.

The restricted reference uses NeoPZ's assembled matrix with a separately imposed macro trace, rather than its historical positive-order MHM controller. See [the comparison case](../../docs/cases/neopz.md) for the spaces, boundary moments, revisions and executed results.

Only archived measurements, field data and figures are read here. Reference sources and comparison execution programs are not included.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
from pathlib import Path
import hashlib
import json

from IPython.display import Image, Markdown, display

RESULTS = ROOT / "examples/results/neopz"
report = json.loads((RESULTS / "comparison.json").read_text())
rows = report["rows"]
for row in rows:
    path = RESULTS / row["archive"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row["archive_sha256"], path.name
print(f"Loaded {len(rows)} records and verified all field archive checksums")


## Analytical problem data

For the affine patch, pressure is p=1+x+2y, permeability is one, source is zero, and physical flux is q=(-1,-2). The mixed pressure is cellwise constant, so its exact representable target is the cell mean, not the pointwise analytical pressure.

The layered cases have permeability contrast 10 or 1000, continuous normal flux -1 and a tangential flux jump from -2 to -2k. That jump is physically admissible in H(div).


In [ ]:
for case in sorted({row["case"] for row in rows}):
    records = [row for row in rows if row["case"] == case]
    levels = sorted({row["macro_resolution"] for row in records})
    traces = sorted({row["trace_segments"] for row in records})
    print(f"{case}: n={levels}, trace segments={traces}, {len(records)} archived results")


## Recorded field differences

The table displays the measurements already obtained at n=2. During the executed study, physical pressure, flux and divergence fields were compared on matched meshes. The stored three-point triangle rule integrates squared RT0 field differences exactly; errors against smooth analytical fields use higher-order quadrature.

Displaying the measurements and checking archive integrity does not constitute a new solver execution.


In [ ]:
small_rows = [row for row in rows if row["macro_resolution"] == 2]
header = "| Case | Trace segments | Pressure difference L2 | Flux difference L2 | Divergence difference L2 |"
lines = [header, "|---|---:|---:|---:|---:|"]
for result in small_rows:
    lines.append(
        f"| {result['case']} | {result['trace_segments']} | "
        f"{result['pressure_difference_l2']:.3e} | "
        f"{result['flux_difference_l2']:.3e} | "
        f"{result['divergence_difference_l2']:.3e} |"
    )
display(Markdown("\n".join(lines)))
print(f"Displayed {len(small_rows)} previously executed comparisons")


## Five-level convergence and the actual macro mesh

The error curves compare both implementations for full and restricted traces. The field figures show analytical, NeoPZ, pyMHM, and signed implementation-difference panels. The first three columns share physical color scales; differences have their own scales.

Every field panel contains the actual macro mesh. Flux is rendered affinely inside each fine triangle without smoothing jumps; pressure remains P0. Analytical cosine fields are sampled for display only, independently of the quadrature used in error norms.

In [ ]:
figures = ROOT / "docs/figures/neopz"
for filename in ("darcy-convergence.png", "cos-fields-n8-s1.png"):
    display(Image(filename=str(figures / filename)))

## Fitted material interfaces

Compare the constant normal flux with the tangential jump at x=0.5. The profile uses y=0.37 and marks all crossings of the actual macro mesh. Its P0 pressure steps are the computed cell values, not an interpolated pressure reconstruction.

In [ ]:
for filename in (
    "layer-fields-n4-s1.png",
    "layer1000-fields-n4-s1.png",
    "layer-profiles.png",
):
    display(Image(filename=str(figures / filename)))

To regenerate these figures from the archived data, run:

    pixi run -e notebooks gallery-neopz

The recorded restricted reference used T-transpose A T from NeoPZ's mixed operator, with matching normal orientations and edge-integral scaling. Its fields were evaluated by NeoPZ after the restricted SciPy solve. The record checks both the local RT0 operators and the reduced MHM trace; this notebook visualizes that completed study.
